# Generator w architekturze RAGOstatni komponent potoku RAG: model językowy, który dostaje ponumerowaną listę wyszukanych dokumentów i pytanie, a zwraca odpowiedź z cytowaniami. Porównanie modelu instrukcyjnego z czatowym, przejście z `pipeline` na natywne API `transformers` oraz FlashAttention-2.

In [ ]:
!pip install -U transformers datasets

In [ ]:
def test_generating(model_name: str, device_map = "auto"):
  # Prompt do generatora z kontekstem opartym na przykładowych wyszukanych tekstach
  prepared_prompt = """
  Numerowana lista dokumentów jest poniżej:
  ---------------------

  Dokument: 0

  Uzyskaj paszport - usługa dla osoby dorosłej
  Co musisz przygotoAwać
  twoje jedno aktualne, kolorowe zdjęcie - zrób je u fotografa nie wcześniej niż 6 miesięcy przed złożeniem wniosku. Jeśli chcesz je zrobić samodzielnie - sprawdź, jak wykonać zdjęcie do paszportu,
  dowód opłaty za paszport (po złożeniu wniosku opłata nie podlega zwrotowi),
  jeśli przysługuje ci prawo do zniżki lub zwolnienia z opłaty - dokument, który to potwierdzi, na przykład ważna legitymacja studencka (szczegóły znajdziesz w sekcji Zwolnienia i zniżki),
  ważny paszport, którego obecnie używasz. Jeśli nie masz ważnego paszportu - weź ze sobą ważny dowód osobisty,
  jeśli twoje nazwisko zostało zmienione po ślubie za granicą - skrócony lub zupełny odpis polskiego aktu małżeństwa.


  Dokument: 1

  Uzyskaj paszport - usługa dla osoby dorosłej
  Jak odbierzesz dokument

  Osobiście w punkcie paszportowym - tam, gdzie złożysz wniosek.

  Weź ze sobą:

  ważny dokument tożsamości,
  aktualny dokument paszportowy - jeśli masz,
  poprzedni paszport - jeśli masz. Zostanie on anulowany.
  Przy odbiorze paszportu sprawdź - czy twoje dane osobowe i biometryczne są poprawne.


  ---------------------
  Odpowiedz na pytanie użytkownika wykorzystując tylko informacje znajdujące się w dokumentach, a nie wcześniejszą wiedzę.
  Udziel wysokiej jakości, poprawnej gramatycznie odpowiedzi w języku polskim. Odpowiedź powinna zawierać cytowania do dokumentów, z których pochodzą informacje. Zacytuj dokument za pomocą symbolu [nr_dokumentu] powołując się na fragment np. [0] dla fragmentu z dokumentu 0. Jeżeli w dokumentach nie ma informacji potrzebnych do odpowiedzi na pytanie, zamiast odpowiedzi zwróć tekst: "Nie udało mi się odnaleźć odpowiedzi na pytanie".

  Pytanie: Jak odebrać paszport?"""

  import transformers
  import torch


  pipeline = transformers.pipeline(
      "text-generation",
      model=model_name,
      model_kwargs={"torch_dtype": torch.float16}, #float16 if not ampere/hopper gpu
      device_map=device_map,
  )

  messages = [
      {"role": "system", "content": "You are an administration chatbot assistant!"},
      {"role": "user", "content": prepared_prompt},
  ]

  outputs = pipeline(
      messages,
      max_new_tokens=256,
  )
  print(outputs[0]["generated_text"][-1])


In [ ]:
model_name = "CYFRAGOVPL/Llama-PLLuM-8B-instruct"
test_generating(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

... (6 outputs left out)

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/525 [00:00<?, ?B/s]

Device set to use cuda:0


{'role': 'assistant', 'content': 'Aby odebrać paszport, należy udać się osobiście do punktu paszportowego, w którym złożono wniosek [1]. Przy odbiorze paszportu należy mieć przy sobie ważny dokument tożsamości, aktualny dokument paszportowy (jeśli jest się w posiadaniu) oraz poprzedni paszport (jeśli się go posiada), który zostanie anulowany [1]. Przy odbiorze paszportu należy również sprawdzić, czy dane osobowe i biometryczne są poprawne [1].'}


## Model instrukcyjny a model czatowyTen sam prompt podany `Llama-PLLuM-8B-instruct` i `Llama-PLLuM-8B-chat`, dodatkowo z `device_map="auto"` i `device_map="cuda"`.

In [ ]:
model_name = "CYFRAGOVPL/Llama-PLLuM-8B-chat"
test_generating(model_name, device_map="auto")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/931 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

... (6 outputs left out)

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/525 [00:00<?, ?B/s]

Device set to use cuda:0


{'role': 'assistant', 'content': 'Aby odebrać paszport, należy udać się do punktu paszportowego, w którym złożono wniosek [1]. Przy odbiorze paszportu należy mieć przy sobie ważny dokument tożsamości, aktualny dokument paszportowy (jeśli jest) oraz poprzedni paszport (jeśli jest), który zostanie anulowany [1]. Przy odbiorze należy również sprawdzić, czy dane osobowe i biometryczne są poprawne [1].'}


In [ ]:
model_name = "CYFRAGOVPL/Llama-PLLuM-8B-chat"
test_generating(model_name, device_map="cuda")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

ValueError: Could not load model CYFRAGOVPL/Llama-PLLuM-8B-chat with any of the following classes: (<class 'transformers.models.auto.modeling_auto.AutoModelForCausalLM'>, <class 'transformers.models.auto.modeling_tf_auto.TFAutoModelForCausalLM'>, <class 'transformers.models.llama.modeling_llama.LlamaForCausalLM'>). See the original errors:

while loading with AutoModelForCausalLM, an error is thrown:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/transformers/pipelines/base.py", line 293, in infer_framework_load_model
    model = model_class.from_pretrained(model, **kwargs)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/auto/auto_factory.py", line 604, in from_pretrained
    return model_class.from_pretrained(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py", line 277, in _wrapper
    return func(*args, 

**Obserwacje**
1. Llama-PLLuM-8B-instruct jest modelem instrukcyjnym, więc jest bardziej precyzyjny, ale też nieco 'robotyczny'. Można to zauważyć w wyrażeniach typu "jeśli się go posiada" ("jeśli jest" użyte przez model chat jest bardziej naturalne) oraz nieo sztucznym dodaniu słówka "paszport" w ostatnim zdaniu. Llama-PLLuM-8B-chat potrafi lepiej prowadzić płynny, bezpieczny i uprzejmy dialog, a jego odpowiedź jest bardziej naturalna.
2. device='cuda' wymusza załadowanie całego modelu do pamięci GPU. Jak widać powyżej, spowodowało to błąd w Google Collab, który posiada limit zużycia GPU 14.74GiB. Błąd nie pojawił się podczas używania device='auto', ponieważ device_map="auto" uruchamia inteligentnego menedżera zasobów, który tworzy mapę rozmieszczenia warstw modelu na dostępnych urządzeniach (GPU, CPU, Dysk).

## Bez `pipeline` — natywne API `transformers``AutoModelForCausalLM` + `AutoTokenizer`, ręczne `apply_chat_template`, `generate` i dekodowanie wyłącznie nowo wygenerowanych tokenów.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [ ]:
def test_generating_native_transformers(model_name: str):
  prepared_prompt = """
  Numerowana lista dokumentów jest poniżej:
  ---------------------

  Dokument: 0

  Uzyskaj paszport - usługa dla osoby dorosłej
  Co musisz przygotować
  twoje jedno aktualne, kolorowe zdjęcie - zrób je u fotografa nie wcześniej niż 6 miesięcy przed złożeniem wniosku. Jeśli chcesz je zrobić samodzielnie - sprawdź, jak wykonać zdjęcie do paszportu,
  dowód opłaty za paszport (po złożeniu wniosku opłata nie podlega zwrotowi),
  jeśli przysługuje ci prawo do zniżki lub zwolnienia z opłaty - dokument, który to potwierdzi, na przykład ważna legitymacja studencka (szczegóły znajdziesz w sekcji Zwolnienia i zniżki),
  ważny paszport, którego obecnie używasz. Jeśli nie masz ważnego paszportu - weź ze sobą ważny dowód osobisty,
  jeśli twoje nazwisko zostało zmienione po ślubie za granicą - skrócony lub zupełny odpis polskiego aktu małżeństwa.


  Dokument: 1

  Uzyskaj paszport - usługa dla osoby dorosłej
  Jak odbierzesz dokument

  Osobiście w punkcie paszportowym - tam, gdzie złożysz wniosek.

  Weź ze sobą:

  ważny dokument tożsamości,
  aktualny dokument paszportowy - jeśli masz,
  poprzedni paszport - jeśli masz. Zostanie on anulowany.
  Przy odbiorze paszportu sprawdź - czy twoje dane osobowe i biometryczne są poprawne.


  ---------------------
  Odpowiedz na pytanie użytkownika wykorzystując tylko informacje znajdujące się w dokumentach, a nie wcześniejszą wiedzę.
  Udziel wysokiej jakości, poprawnej gramatycznie odpowiedzi w języku polskim. Odpowiedź powinna zawierać cytowania do dokumentów, z których pochodzą informacje. Zacytuj dokument za pomocą symbolu [nr_dokumentu] powołując się na fragment np. [0] dla fragmentu z dokumentu 0. Jeżeli w dokumentach nie ma informacji potrzebnych do odpowiedzi na pytanie, zamiast odpowiedzi zwróć tekst: "Nie udało mi się odnaleźć odpowiedzi na pytanie".

  Pytanie: Jak odebrać paszport?"""
  messages = [
        {"role": "system", "content": "You are an administration chatbot assistant!"},
        {"role": "user", "content": prepared_prompt},
    ]

  tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")
  tokenizer.pad_token = tokenizer.eos_token

  model = AutoModelForCausalLM.from_pretrained(
      model_name,
      device_map="auto",
      torch_dtype=torch.float16,
      low_cpu_mem_usage=True)

  model_inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

  model_output = model.generate(
      **model_inputs,
      max_new_tokens=256,
      pad_token_id=tokenizer.eos_token_id
  )
  input_length = model_inputs["input_ids"].shape[1]
  new_tokens = model_output[:, input_length:]

  decoded_response = tokenizer.decode(
      new_tokens[0],
      skip_special_tokens=True
  )
  return decoded_response



In [ ]:
test_generating_native_transformers("CYFRAGOVPL/Llama-PLLuM-8B-instruct")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/525 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


... (5 outputs left out)

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

'Aby odebrać paszport, należy osobiście udać się do punktu paszportowego, w którym złożono wniosek [1]. Przy odbiorze paszportu należy mieć przy sobie ważny dokument tożsamości, aktualny dokument paszportowy (jeśli posiada się taki dokument) oraz poprzedni paszport (jeśli posiada się taki dokument), który zostanie anulowany [1]. Przy odbiorze paszportu należy sprawdzić, czy dane osobowe i biometryczne są poprawne [1].'

## FlashAttention-2Ta sama generacja z zoptymalizowaną implementacją atencji.

In [ ]:
def test_generating_native_transformers_optimized(model_name: str):
  prepared_prompt = """
  Numerowana lista dokumentów jest poniżej:
  ---------------------

  Dokument: 0

  Uzyskaj paszport - usługa dla osoby dorosłej
  Co musisz przygotować
  twoje jedno aktualne, kolorowe zdjęcie - zrób je u fotografa nie wcześniej niż 6 miesięcy przed złożeniem wniosku. Jeśli chcesz je zrobić samodzielnie - sprawdź, jak wykonać zdjęcie do paszportu,
  dowód opłaty za paszport (po złożeniu wniosku opłata nie podlega zwrotowi),
  jeśli przysługuje ci prawo do zniżki lub zwolnienia z opłaty - dokument, który to potwierdzi, na przykład ważna legitymacja studencka (szczegóły znajdziesz w sekcji Zwolnienia i zniżki),
  ważny paszport, którego obecnie używasz. Jeśli nie masz ważnego paszportu - weź ze sobą ważny dowód osobisty,
  jeśli twoje nazwisko zostało zmienione po ślubie za granicą - skrócony lub zupełny odpis polskiego aktu małżeństwa.


  Dokument: 1

  Uzyskaj paszport - usługa dla osoby dorosłej
  Jak odbierzesz dokument

  Osobiście w punkcie paszportowym - tam, gdzie złożysz wniosek.

  Weź ze sobą:

  ważny dokument tożsamości,
  aktualny dokument paszportowy - jeśli masz,
  poprzedni paszport - jeśli masz. Zostanie on anulowany.
  Przy odbiorze paszportu sprawdź - czy twoje dane osobowe i biometryczne są poprawne.


  ---------------------
  Odpowiedz na pytanie użytkownika wykorzystując tylko informacje znajdujące się w dokumentach, a nie wcześniejszą wiedzę.
  Udziel wysokiej jakości, poprawnej gramatycznie odpowiedzi w języku polskim. Odpowiedź powinna zawierać cytowania do dokumentów, z których pochodzą informacje. Zacytuj dokument za pomocą symbolu [nr_dokumentu] powołując się na fragment np. [0] dla fragmentu z dokumentu 0. Jeżeli w dokumentach nie ma informacji potrzebnych do odpowiedzi na pytanie, zamiast odpowiedzi zwróć tekst: "Nie udało mi się odnaleźć odpowiedzi na pytanie".

  Pytanie: Jak odebrać paszport?"""
  messages = [
        {"role": "system", "content": "You are an administration chatbot assistant!"},
        {"role": "user", "content": prepared_prompt},
    ]

  tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")
  tokenizer.pad_token = tokenizer.eos_token

  model = AutoModelForCausalLM.from_pretrained(
      model_name,
      device_map="auto",
      torch_dtype=torch.float16,
      low_cpu_mem_usage=True,
      attn_implementation="sdpa")

  model_inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

  model_output = model.generate(
      **model_inputs,
      max_new_tokens=256,
      pad_token_id=tokenizer.eos_token_id
  )
  input_length = model_inputs["input_ids"].shape[1]
  new_tokens = model_output[:, input_length:]

  decoded_response = tokenizer.decode(
      new_tokens[0],
      skip_special_tokens=True
  )
  return decoded_response



In [ ]:
test_generating_native_transformers_optimized("CYFRAGOVPL/Llama-PLLuM-8B-instruct")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/525 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


... (5 outputs left out)

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

'Aby odebrać paszport, należy osobiście zgłosić się do punktu paszportowego, w którym złożono wniosek [1]. Przy odbiorze paszportu należy mieć przy sobie ważny dokument tożsamości, aktualny dokument paszportowy (jeśli jest się w posiadaniu), a także poprzedni paszport (jeśli jest się w posiadaniu), który zostanie anulowany [1]. Przy odbiorze paszportu należy sprawdzić, czy dane osobowe i biometryczne są poprawne [1].'

**Obserwacje**

Zauważyłam wzrost szybkości generacji (8 min zamiast 11 min). Nie jest to drastyczny skok wydajności, ponieważ tekst wejściowy nie był długi. Przy krótkich sekwencjach (promptach), obliczanie samej atencji zajmuje tylko ułamek całkowitego czasu pracy modelu.

Zauważyłam też spadek zużycia pamięci VRAM. Był on niewielki, również ze względu na krótką sekwencję (przy małym "N", macierz atencji nie zajmuje dużo miejsca).

Oszczędność ta wynika z faktu, że zastosowana optymalizacja SDPA (podobnie jak Flash Attention) nie zapisuje wielkiej macierzy N×N w pamięci głównej GPU. Zamiast tego obliczenia są dzielone na mniejsze bloki ("kafelki"), które są przetwarzane bezpośrednio w szybkiej pamięci podręcznej (SRAM) układu graficznego, co redukuje kosztowny transfer danych.